# P95 — Las siete herramientas de la inferencia causal, con reflexiones sobre aprendizaje automático

## 1. Título y paper

**Paper:** *The Seven Tools of Causal Inference, with Reflections on Machine Learning*  
**Autoría:** Judea Pearl  
**Año y venue:** 2019 · Communications of the ACM, 62(3), 54–60  
**Nivel:** L3 · **Motor:** `causalidad`  
**Ficha completa:** [`P95_causalidad`](../../papers/foundational/P95_causalidad/README.md)

**Hito:** Ordena en tres peldaños lo que un sistema puede responder —asociación, intervención y contrafáctico— y muestra que subir de peldaño exige supuestos que los datos no contienen.

- [doi:10.1145/3241036](https://doi.org/10.1145/3241036)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: El aprendizaje automático ajusta funciones sobre distribuciones observadas, y con eso responde preguntas de asociación. Pero las decisiones que importan son de intervención —«¿qué pasa si hago X?»— y esa pregunta no se puede responder solo con datos observacionales, por muchos que sean.
2. Ejecutar una implementación mínima de la propuesta: La escalera de la causalidad y siete herramientas asociadas: modelos gráficos, el operador do, el criterio de puerta trasera, la fórmula de ajuste, mediación, transportabilidad y datos faltantes. La estructura causal se declara; no se estima de la tabla.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P91
- P80
- Rubin (1974), resultados potenciales


## 4. Intuición

Un tratamiento funciona en los pacientes leves. Funciona en los graves. Y en la tabla completa, aparentemente, perjudica. No hay error de cálculo: los tres números son correctos. Ningún análisis de los datos decide cuál hay que creer, y esa es la tesis.


## 5. Concepto mínimo

```text
Escalera de la causalidad:
  1. ASOCIACIÓN     P(Y | X)              ¿qué me dice ver X sobre Y?
  2. INTERVENCIÓN   P(Y | do(X))          ¿qué pasa si HAGO X?
  3. CONTRAFÁCTICO  P(Y_x | X', Y')       ¿qué habría pasado si...?

Subir de peldaño exige supuestos que NO están en los datos: el grafo causal.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('causalidad', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Gana el tratamiento en cada subgrupo?
2. ¿Y en el agregado?
3. ¿Cuál de las dos lecturas es la correcta?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('causalidad', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('causalidad', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Gana en los leves (0,931 frente a 0,8667) y gana en los graves (0,73 frente a 0,6875). Y en el agregado **pierde**: 0,78 frente a 0,8257. Es la paradoja de Simpson. Ajustando por la gravedad —un confusor, porque influye en quién recibe tratamiento **y** en el resultado— el efecto de intervenir es 0,8325 frente a 0,7789: el signo se invierte otra vez.


## 10. Comentario pedagógico

La respuesta a la tercera pregunta no está en los datos. Depende de si la gravedad es causa del tratamiento —en cuyo caso hay que ajustar— o consecuencia suya —en cuyo caso ajustar es el error—. Los números son idénticos en ambos casos. Hace falta declarar el grafo, y esa declaración viene de fuera de la tabla.


## 11. Error o anti-patrón deliberado

Anti-patrón: resolver la paradoja mirando más datos o afinando el modelo.


In [ ]:
print('Con mas datos la paradoja se vuelve mas nitida, no desaparece.')
print('Con un modelo mejor, tambien: ninguno de los dos puede saber que causa que.')
print('Ese supuesto se declara, se discute y se defiende. No se estima.')

## 12. Corrección

El procedimiento correcto, con los tres peldaños separados:


In [ ]:
r = run_paper_lab('causalidad', seed=7)['result']
for peldano, contenido in r['escalera_de_la_causalidad'].items():
    print(f"{peldano}: {contenido['pregunta']}")
    print(f"   operacion: {contenido['operacion']}")
print()
print('sin ajustar :', r['agregado_sin_ajustar'])
print('tras ajustar:', r['tras_ajustar_por_gravedad'])

## 13. Desafío guiado

Explica por qué el tercer peldaño —los contrafácticos— no se puede calcular con estos datos ni con el grafo.


In [ ]:
r = run_paper_lab('causalidad', seed=3)['result']
show(r)

## 14. Desafío autónomo

Busca en tu trabajo una decisión que se justifique con una correlación. Dibuja el grafo causal que estás suponiendo sin decirlo, identifica los confusores y comprueba si el ajuste cambia la conclusión.


## 15. Evidencia de aprendizaje

Guarda la tabla por subgrupos, el agregado y el ajuste, con tu grafo causal explícito.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P95_causalidad/README.md) · evaluación formal: [`assessments/papers/P95_causalidad.md`](../../assessments/papers/P95_causalidad.md)


## 16. Cierre

Aquí se cierra la ruta probabilística. Lo siguiente sale de la pantalla: sistemas que perciben y actúan sobre el mundo físico, donde equivocarse tiene consecuencias que no se deshacen.


## 17. Conexión con el siguiente hito

- P72

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
